# Zara Brand Sentiment Classifier

This notebook implements the PE6201 pipeline: classify comments into five operational categories with DeepSeek, calibrate an abstention threshold, compare a keyword baseline, detect risk shifts, and generate a Markdown briefing. The 30-row real set is held out from tuning.

In [5]:
# Google Colab only: run once if packages are missing
# !pip -q install pandas requests scikit-learn matplotlib tabulate

from pathlib import Path
import json, os, re, time
import pandas as pd

from google.colab import drive
drive.mount("/content/drive")

# 数据集所在的文件夹（两个 CSV 都放这里）
DATA_ROOT = "/content/drive/MyDrive/zara"
DATA_DIR = Path(DATA_ROOT)

LABELS = ['quality_complaint', 'logistics_complaint', 'competitor_comparison', 'price_sensitive', 'other']
MAX_COMMENTS_PER_RUN = 250

# 检查两个 CSV 是否都在 zara 文件夹里
assert (DATA_DIR / 'synthetic_reviews.csv').exists(), \
    f'找不到 {DATA_DIR}/synthetic_reviews.csv，请把 CSV 上传到 Google Drive 的 zara 文件夹'
assert (DATA_DIR / 'real_eval.csv').exists(), \
    f'找不到 {DATA_DIR}/real_eval.csv，请把 CSV 上传到 Google Drive 的 zara 文件夹'

print("data file found：")
print("  -", DATA_DIR / 'synthetic_reviews.csv')
print("  -", DATA_DIR / 'real_eval.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
data file found：
  - /content/drive/MyDrive/zara/synthetic_reviews.csv
  - /content/drive/MyDrive/zara/real_eval.csv


## 1 Load and validate data

The synthetic set is development data. The real set must never be used to modify prompts or choose the threshold.

In [6]:
synthetic = pd.read_csv(DATA_DIR / 'synthetic_reviews.csv')
real_eval = pd.read_csv(DATA_DIR / 'real_eval.csv')

def validate_dataset(df, name, required):
    assert set(required).issubset(df.columns), f'{name}: missing columns'
    assert df['comment_text'].notna().all() and df['comment_text'].str.strip().ne('').all(), f'{name}: empty comment'
    assert df['true_category'].isin(LABELS).all(), f'{name}: invalid label'
    assert not df['comment_text'].duplicated().any(), f'{name}: duplicate comment text'

validate_dataset(synthetic, 'synthetic', ['comment_id','comment_text','platform','true_category','batch_id'])
validate_dataset(real_eval, 'real_eval', ['comment_id','comment_text','platform','true_category','source_url'])
assert len(synthetic) == 200 and synthetic['true_category'].value_counts().eq(40).all()
assert len(real_eval) == 30
print('Synthetic distribution:')
print(synthetic['true_category'].value_counts().sort_index())
print('Real evaluation distribution:')
print(real_eval['true_category'].value_counts().sort_index())

Synthetic distribution:
true_category
competitor_comparison    40
logistics_complaint      40
other                    40
price_sensitive          40
quality_complaint        40
Name: count, dtype: int64
Real evaluation distribution:
true_category
competitor_comparison    6
logistics_complaint      6
other                    6
price_sensitive          6
quality_complaint        6
Name: count, dtype: int64


## 2 DeepSeek structured classifier

Set `DEEPSEEK_API_KEY` as a Colab secret or environment variable. The function sends no usernames or account identifiers.

In [7]:
import os, re, json, time, requests

SYSTEM_PROMPT = '''You are Zara's brand social-media comment classifier.
Classify ONE comment into exactly one label: quality_complaint, logistics_complaint, competitor_comparison, price_sensitive, other.
Definitions: quality_complaint = material, sizing, damage, durability or defect; logistics_complaint = delivery, tracking, parcel, return-shipping or fulfillment; competitor_comparison = explicitly compares Zara with another brand; price_sensitive = price, discount, value or affordability; other = all remaining content, including praise, questions and customer-service-only complaints.
Return only valid JSON: {"category":"one label","confidence":0.0}. Confidence must be from 0 to 1.'''

def extract_json(text):
    match = re.search(r'\{.*?\}', text, re.S)
    if not match:
        raise ValueError('Model response did not contain JSON')
    return json.loads(match.group(0))

def classify_comment(comment_text, api_key=None, retries=3):
    # get OpenRouter API Key
    api_key = api_key or os.environ.get("OPENROUTER_API_KEY")
    if not api_key:
        try:
            from google.colab import userdata
            api_key = userdata.get("OPENROUTER_API_KEY")
        except Exception:
            api_key = None
    if not api_key:
        raise EnvironmentError('Set OPENROUTER_API_KEY in environment or Colab secrets.')

    payload = {
        'model': 'deepseek/deepseek-chat',
        'temperature': 0,
        'max_tokens': 100,
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': comment_text}
        ]
    }
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Content-Type': 'application/json',
        # two header optional，but OpenRouter recommend to add
        'HTTP-Referer': 'https://colab.research.google.com',
        'X-Title': 'Zara Sentiment Classifier'
    }

    for attempt in range(retries):
        try:
            response = requests.post(
                'https://openrouter.ai/api/v1/chat/completions',
                headers=headers,
                json=payload,
                timeout=45
            )
            response.raise_for_status()
            parsed = extract_json(response.json()['choices'][0]['message']['content'])
            category = parsed.get('category', 'other')
            confidence = float(parsed.get('confidence', 0))
            return {
                'predicted_category': category if category in LABELS else 'other',
                'confidence': min(1, max(0, confidence))
            }
        except Exception as error:
            if attempt == retries - 1:
                return {'predicted_category': 'other', 'confidence': 0.0, 'error': str(error)}
            time.sleep(2 ** attempt)

## 3 Run classification and save an auditable log

In [14]:
def classify_dataset(df):
    if len(df) > MAX_COMMENTS_PER_RUN:
        raise ValueError(f'Maximum is {MAX_COMMENTS_PER_RUN} comments per run.')
    unique = df.drop_duplicates('comment_text').copy()
    predictions = [classify_comment(text) for text in unique['comment_text']]
    result = pd.concat([unique.reset_index(drop=True), pd.DataFrame(predictions)], axis=1)
    result.to_csv('classification_log.csv', index=False, encoding='utf-8-sig')
    return result

# Run development classification first. Keep real_eval untouched until the final evaluation.
synthetic_predictions = classify_dataset(synthetic)
synthetic_predictions.head()

,comment_id,comment_text,platform,true_category,batch_id,source_type,predicted_category,confidence,error
0,1,The Zara blazer started fraying after one wash...,Xiaohongshu,quality_complaint,week1,synthetic,quality_complaint,0.95,NaN
1,2,The Zara dress started pilling after two wears...,Instagram,quality_complaint,week2,synthetic,quality_complaint,0.95,NaN
2,3,The Zara trousers started shrinking after thre...,TikTok,quality_complaint,week3,synthetic,quality_complaint,0.90,NaN
3,4,The Zara shirt started coming apart after a we...,Xiaohongshu,quality_complaint,week1,synthetic,quality_complaint,0.95,NaN
4,5,The Zara jacket started losing shape after ten...,Instagram,quality_complaint,week2,synthetic,quality_complaint,0.95,NaN


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


## 4 Evaluation, calibration, and keyword baseline

In [15]:
KEYWORDS = {
    'quality_complaint': ['broken', 'hole', 'shrink', 'fray', 'pilling', 'defect', 'damage'],
    'logistics_complaint': ['delivery', 'shipping', 'parcel', 'tracking', 'transit', 'delivered', 'return parcel'],
    'competitor_comparison': ['h&m', 'uniqlo', 'mango', 'cos', 'temu', 'amazon', 'compared'],
    'price_sensitive': ['price', 'expensive', 'discount', 'sale', 'coupon', 'value', 'budget'],
}

def keyword_baseline(text):
    lower = text.lower()
    scores = {label: sum(word in lower for word in words) for label, words in KEYWORDS.items()}
    best = max(scores, key=scores.get)
    return best if scores[best] else 'other'

def calibration_table(predictions):
    rows = []
    for threshold in [round(x / 10, 1) for x in range(3, 10)]:
        kept = predictions[predictions['confidence'] >= threshold]
        rows.append({'threshold': threshold, 'abstention_rate': 1 - len(kept)/len(predictions), 'non_abstained_accuracy': (kept['predicted_category'] == kept['true_category']).mean() if len(kept) else float('nan')})
    return pd.DataFrame(rows)

def choose_threshold(table):
    eligible = table[table['abstention_rate'] <= 0.15]
    return float((eligible if not eligible.empty else table).sort_values('non_abstained_accuracy', ascending=False).iloc[0]['threshold'])

def evaluate(predictions, threshold):
    out = predictions.copy()
    out['abstained'] = out['confidence'] < threshold
    out['final_category'] = out['predicted_category'].where(~out['abstained'], 'uncertain')
    kept = out[~out['abstained']]
    out['baseline_category'] = out['comment_text'].map(keyword_baseline)
    metrics = pd.DataFrame([{
        'n': len(out), 'threshold': threshold, 'abstention_rate': out['abstained'].mean(),
        'non_abstained_accuracy': (kept['predicted_category'] == kept['true_category']).mean() if len(kept) else float('nan'),
        'forced_choice_accuracy': (out['predicted_category'] == out['true_category']).mean(),
        'keyword_baseline_accuracy': (out['baseline_category'] == out['true_category']).mean(),
    }])
    return out, metrics

# Calibration must use a development slice only, never real_eval.
calibration_dev = synthetic_predictions.sample(20, random_state=6201)
table = calibration_table(calibration_dev)
THRESHOLD = choose_threshold(table)
table

,threshold,abstention_rate,non_abstained_accuracy
0,0.3,0.0,0.95
1,0.4,0.0,0.95
2,0.5,0.0,0.95
3,0.6,0.0,0.95
4,0.7,0.0,0.95
5,0.8,0.0,0.95
6,0.9,0.0,0.95


## 5 Shift alerts and management briefing

In [16]:
def alert_for_batch(batch_df, previous_pct=None):
    pct = batch_df['final_category'].value_counts(normalize=True).reindex(LABELS, fill_value=0)
    uncertain_rate = (batch_df['final_category'] == 'uncertain').mean()
    other_rate = pct['other']
    shifts = (pct - previous_pct) if previous_pct is not None else pd.Series(0.0, index=LABELS)
    if uncertain_rate > 0.20:
        status, message = 'YELLOW', 'Review required: uncertainty exceeds 20%.'
    elif other_rate > 0.30:
        status, message = 'YELLOW', 'Model drift alert: other exceeds 30%.'
    elif (shifts >= 0.10).any():
        changed = ', '.join(shifts[shifts >= 0.10].index)
        status, message = 'YELLOW', f'Risk shift: {changed} rose by at least 10 percentage points.'
    else:
        status, message = 'GREEN', 'No material category shift detected.'
    return status, message, pct

def make_report(predictions, metrics):
    lines = ['# Zara Brand Sentiment Briefing', '', '## Evaluation', metrics.to_csv(index=False), '', '## Batch alerts']
    previous = None
    for batch, frame in predictions.groupby('batch_id', sort=True):
        status, message, previous = alert_for_batch(frame, previous)
        lines += [f'### {batch}: {status}', message, '']
    Path('Alert_Report.md').write_text('\n'.join(lines), encoding='utf-8')

# After classification and calibration:
synthetic_scored, synthetic_metrics = evaluate(synthetic_predictions, THRESHOLD)
synthetic_scored.to_csv('predicted_labels.csv', index=False, encoding='utf-8-sig')
synthetic_metrics.to_csv('eval_results.csv', index=False, encoding='utf-8-sig')
make_report(synthetic_scored, synthetic_metrics)

## 6 Final held-out test

Run this only after the prompt and threshold are frozen. The model call produces `real_eval_predictions.csv`, and results are reported separately from synthetic performance.

In [18]:
real_predictions = classify_dataset(real_eval)
real_scored, real_metrics = evaluate(real_predictions, THRESHOLD)
real_scored.to_csv('real_eval_predictions.csv', index=False, encoding='utf-8-sig')
print(real_metrics.to_string(index=False))
print('Most frequently abstained comments:')
print(real_scored[real_scored['abstained']][['comment_text','true_category']].head(10).to_string(index=False))
real_metrics.to_csv('real_eval_results.csv', index=False, encoding='utf-8-sig')

 n  threshold  abstention_rate  non_abstained_accuracy  forced_choice_accuracy  keyword_baseline_accuracy
30        0.3              0.0                     0.8                     0.8                        0.5
Most frequently abstained comments:
Empty DataFrame
Columns: [comment_text, true_category]
Index: []


## Responsible-use checklist

- Keep source URLs but remove usernames, avatars, timestamps, order numbers, and addresses.
- Use results to prioritize human review; do not make an automated crisis decision.
- Re-label real comments manually before final submission and report mistakes honestly.
- The supplied real set is from public Reddit and Trustpilot pages, not falsely presented as Xiaohongshu/Instagram/TikTok data. If the rubric specifically requires those platforms, replace it with manually collected public comments that comply with each platform's terms.